# PJM RT Load Comparison

Compare RT Metered, RT Prelim, and RT Instantaneous load across regions.

In [2]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Add backend/src to path so we can import the utils
REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))

from src.utils.azure_postgresql import pull_from_db

# ── Target date ──
TARGET_DATE = "2026-02-26"

In [3]:
# Read SQL query from file and inject target date
sql_path = Path.cwd() / "pjm_load_rt_comparison.sql"
query = sql_path.read_text().format(target_date=TARGET_DATE)

# Pull data
df = pull_from_db(query=query)
print(f"{len(df):,} rows | Target date: {TARGET_DATE}")
print(f"Regions: {sorted(df['region'].unique())}")
print(f"Load types: {sorted(df['load_type'].unique())}")
df.head()

288 rows | Target date: 2026-02-26
Regions: ['MIDATL', 'RTO', 'SOUTH', 'WEST']
Load types: ['instantaneous', 'metered', 'prelim']


,datetime,date,hour_ending,region,rt_load_mw,load_type
0,2026-02-26 01:00:00,2026-02-26,1.0,MIDATL,29278.276017,instantaneous
1,2026-02-26 01:00:00,2026-02-26,1.0,MIDATL,29103.648000,metered
2,2026-02-26 01:00:00,2026-02-26,1.0,MIDATL,29243.600000,prelim
3,2026-02-26 01:00:00,2026-02-26,1.0,RTO,91514.823383,instantaneous
4,2026-02-26 01:00:00,2026-02-26,1.0,RTO,90889.190000,metered


In [4]:
df.dtypes

datetime       datetime64[us]
date                   object
hour_ending           float64
region                    str
rt_load_mw            float64
load_type                 str
dtype: object

## RT Load by Load Type (per Region)

In [12]:
sorted(df["region"].unique())

['MIDATL', 'RTO', 'SOUTH', 'WEST']

In [13]:
# for region in sorted(df["region"].unique()):
for region in ['RTO', 'WEST']:
    df_region = df[df["region"] == region]
    fig = px.line(
        df_region,
        x="datetime",
        y="rt_load_mw",
        color="load_type",
        title=f"RT Load Comparison - {region}",
        labels={"rt_load_mw": "RT Load (MW)", "datetime": "Date/Hour", "load_type": "Load Type"},
    )
    fig.update_layout(template="plotly_dark", height=500)
    fig.show()

## Differences Between Load Types (per Region)

Pivot to compare metered vs prelim vs instantaneous side-by-side and compute differences.

In [6]:
# Pivot so each load_type becomes a column
df_pivot = df.pivot_table(
    index=["datetime", "date", "hour_ending", "region"],
    columns="load_type",
    values="rt_load_mw",
).reset_index()

# Compute differences where both columns exist
load_types = [c for c in ["metered", "prelim", "instantaneous"] if c in df_pivot.columns]
if "metered" in load_types and "prelim" in load_types:
    df_pivot["metered_minus_prelim"] = df_pivot["metered"] - df_pivot["prelim"]
if "metered" in load_types and "instantaneous" in load_types:
    df_pivot["metered_minus_instantaneous"] = df_pivot["metered"] - df_pivot["instantaneous"]
if "prelim" in load_types and "instantaneous" in load_types:
    df_pivot["prelim_minus_instantaneous"] = df_pivot["prelim"] - df_pivot["instantaneous"]

df_pivot.head()

load_type,datetime,date,hour_ending,region,instantaneous,metered,prelim,metered_minus_prelim,metered_minus_instantaneous,prelim_minus_instantaneous
0,2026-02-26 01:00:00,2026-02-26,1.0,MIDATL,29278.276017,29103.648,29243.6,-139.952,-174.628017,-34.676017
1,2026-02-26 01:00:00,2026-02-26,1.0,RTO,91514.823383,90889.190,91453.7,-564.510,-625.633383,-61.123383
2,2026-02-26 01:00:00,2026-02-26,1.0,SOUTH,14505.666083,14383.315,14498.5,-115.185,-122.351083,-7.166083
3,2026-02-26 01:00:00,2026-02-26,1.0,WEST,46131.377175,47402.227,47711.6,-309.373,1270.849825,1580.222825
4,2026-02-26 02:00:00,2026-02-26,2.0,MIDATL,28688.806367,28486.428,28651.9,-165.472,-202.378367,-36.906367


In [7]:
# Plot differences per region
diff_cols = [c for c in df_pivot.columns if "minus" in c]

for region in sorted(df_pivot["region"].unique()):
    df_r = df_pivot[df_pivot["region"] == region]

    if not diff_cols:
        print(f"No overlapping load types to compare for {region}")
        continue

    fig = make_subplots(
        rows=len(diff_cols), cols=1,
        shared_xaxes=True,
        subplot_titles=[c.replace("_", " ").title() for c in diff_cols],
        vertical_spacing=0.08,
    )

    for i, col in enumerate(diff_cols, start=1):
        fig.add_trace(
            go.Bar(
                x=df_r["datetime"],
                y=df_r[col],
                name=col.replace("_", " ").title(),
            ),
            row=i, col=1,
        )
        fig.update_yaxes(title_text="MW", row=i, col=1)

    fig.update_layout(
        title=f"RT Load Differences - {region}",
        template="plotly_dark",
        height=300 * len(diff_cols),
        showlegend=False,
    )
    fig.show()

## Summary Statistics by Region and Load Type

In [8]:
summary = df.groupby(["region", "load_type"])["rt_load_mw"].agg(
    ["count", "mean", "std", "min", "max"]
).round(1)
summary

count     mean     std      min       max
region load_type                                               
MIDATL instantaneous     24  32236.2  2257.3  28420.3   35530.9
       metered           24  32012.0  2223.0  28252.3   35232.9
       prelim            24  32235.1  2258.3  28416.9   35555.0
RTO    instantaneous     24  99067.3  5532.9  89512.6  106994.0
       metered           24  98527.9  5540.5  88927.4  106438.8
       prelim            24  99076.4  5530.5  89495.2  107070.9
SOUTH  instantaneous     24  15732.3   871.4  14114.5   16806.6
       metered           24  15626.7   869.8  14019.7   16679.2
       prelim            24  15739.8   871.2  14117.7   16803.1
WEST   instantaneous     24  49416.0  2500.0  45379.5   53333.5
       metered           24  50889.2  2610.5  46655.4   54895.6
       prelim            24  51101.5  2563.5  46960.6   55041.6

## Difference Summary Statistics

In [11]:
if diff_cols:
    diff_summary = df_pivot.groupby("region")[diff_cols].agg(
        ["mean", "std", "min", "max"]
    ).round(2)
else:
    print("No differences to summarize.")
diff_summary

load_type metered_minus_prelim                         \
                          mean    std     min     max   
region                                                  
MIDATL                 -223.14  68.36 -401.24 -139.95   
RTO                    -548.44  71.34 -717.39 -440.58   
SOUTH                  -113.08  16.61 -149.72  -84.53   
WEST                   -212.22  68.41 -317.18 -103.46   

load_type metered_minus_instantaneous                            \
                                 mean     std      min      max   
region                                                            
MIDATL                        -224.20   90.46  -475.22   -97.86   
RTO                           -539.39  123.40  -858.41  -283.73   
SOUTH                         -105.56   29.03  -161.40   -40.48   
WEST                          1473.25  161.85  1197.07  1757.73   

load_type prelim_minus_instantaneous                            
                                mean     std      min      max  
region                                                          
MIDATL                         -1.07   50.79   -73.98   117.03  
RTO                             9.05  112.65  -141.03   304.66  
SOUTH                           7.53   16.61   -20.94    52.58  
WEST                         1685.47  111.48  1487.99  1880.26